[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_hoja_camion_v2.ipynb)

# Hoja del camión: ¿con qué umbral mandamos alerta?

**Al terminar podrás** escoger el umbral de alerta de una regresión logística a partir de un presupuesto,
y reportar lo que cuesta en desbalances perdidos y en viajes en balde, con precision y recall.

---

**Para:** equipo de análisis<br>
**De:** Operaciones MiBici, zona centro<br>
**Asunto:** umbral de alerta para el camión de rebalanceo

Tenemos camión para atender **una de cada cinco horas (20 %)** de las estaciones que monitoreamos.
Necesitamos saber:

1. ¿Con qué umbral mandamos la alerta?
2. Con ese umbral, ¿cuántos desbalances atrapamos y cuántos viajes son en balde?

Nos sirve la respuesta en tres renglones, con los números de julio.

---

**Cómo trabajar (30 a 40 min).** Una persona comparte pantalla; las demás dictan. Corran todo en orden.
Solo se editan dos líneas, en el paso 2. En cada pregunta de opción múltiple, escriban la letra en la hoja, debajo de cada pregunta (doble clic para escribir).

| Paso | Qué hacen | Minutos |
|---|---|---|
| 1. Los datos | correr y mirar la gráfica | 5 |
| 2. El modelo | correr; editar `estacion` y `umbral` | 5 |
| 3. La medición | matriz, precision, recall, alertas; el error y el día par | 15 |
| 4. La respuesta | tres renglones | 5 |

Corre las dos celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, precision_score, recall_score
from dammg_utils import COLORES

du.estilo_dammg()

## Paso 1. Los datos

Cada fila es una estación en una hora. La etiqueta $y$ vale **1** si en esa hora y las dos siguientes la estación
tuvo un **desbalance fuerte**: 7 bicis o más de diferencia entre salidas y llegadas, para un lado o para el otro.
Si vale 1, el camión debió ir.

Usamos **train** (octubre a junio) para ajustar y **julio** para medir y escoger el umbral.

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"].copy()   # octubre a junio: para ajustar
julio = df[df.particion == "val"].copy()     # julio: para medir y escoger el umbral

corte = train.future_net.abs().quantile(0.85)   # 7 bicis: el corte de la etiqueta, fijado con train
for d in (train, julio):
    d["y"] = (d.future_net.abs() >= corte).astype(int)   # 1 = desbalance fuerte en las próximas 3 horas
    d["x"] = d.net_lag_168h.abs()                         # desbalance de esa hora, hace una semana (sin signo)

print(f"train: {len(train):,} horas   julio: {len(julio):,} horas   corte: {corte:.0f} bicis")
print(f"horas con desbalance fuerte:  train {train.y.mean():.1%}   julio {julio.y.mean():.1%}")

In [ ]:
# ¿A qué hora del día hay más desbalances? Porcentaje de horas de train con y = 1
por_hora = train.groupby("hour_of_day").y.mean().reset_index()
ax = sns.barplot(data=por_hora, x="hour_of_day", y="y", color=COLORES["w"])
ax.axhline(0.20, color=COLORES["error"], lw=2, ls="--", label="20 %: lo que alcanza el camión")
ax.set(xlabel="hora del día", ylabel="horas con desbalance fuerte", title="Train: desbalances por hora")
ax.legend()
plt.show()

**Qué se ve.** En train hubo desbalance fuerte en el 15.9 % de las horas, y en julio en el **14.5 %**.
En la madrugada casi nunca pasa; en la mañana (7 a 9) y en la tarde (17 a 20) pasa en más de una de cada cinco horas.
Si el camión alcanza para el 20 % de las horas, no alcanza para ir a todas las horas pico de todas las estaciones:
hay que escoger.

## Paso 2. El modelo

Es la logística de la sección 3, con tres datos de cada hora: el desbalance de hace una semana (`x`),
la hora del día y la estación (las dos en one-hot). Tarda unos 10 segundos.

In [ ]:
def columnas(d):
    """x, la hora del día en one-hot y la estación en one-hot."""
    hora = pd.get_dummies(d.hour_of_day, prefix="hora", dtype=int)
    est = pd.get_dummies(d.station_id, prefix="est", dtype=int)
    return pd.concat([d[["x"]], hora, est], axis=1)

modelo = LogisticRegression(max_iter=1000).fit(columnas(train), train.y)
julio["p"] = modelo.predict_proba(columnas(julio))[:, 1]   # p̂: probabilidad de desbalance, hora por hora

julio[["station_id", "hour", "x", "p", "y"]].head()

**Ahora editen estas dos líneas.** `estacion` puede ser `"todas"` o el número de una estación
(por ejemplo 51, 11, 52, 69, 194 o 268). `umbral` es el $t$: mandamos alerta si $\hat p \ge t$.
Empiecen con `"todas"` y `0.5`; después prueben otros valores y vuelvan a correr del paso 2 en adelante.

In [ ]:
estacion = "todas"   # "todas" o el número de una estación, por ejemplo 51
umbral = 0.5         # alerta si p̂ >= umbral

In [ ]:
# Las horas de julio que vamos a medir
if estacion == "todas":
    sel = julio
else:
    sel = julio[julio.station_id == estacion]
alerta = (sel.p >= umbral).astype(int)   # 1 = mandamos camión
print(f"estación: {estacion}   umbral: {umbral}   horas: {len(sel):,}")

## Paso 3. La medición

Cada hora cae en una de cuatro casillas. Filas: lo que **pasó**. Columnas: lo que **dijo** el modelo.

| | dijo calma (no mando) | dijo alerta (mando camión) |
|---|---|---|
| **pasó calma** | TN: bien | FP: viaje en balde |
| **pasó desbalance** | FN: desbalance perdido | TP: llegó el camión |

- **precision** = TP / (TP + FP): de mis alertas, ¿cuántas eran ciertas?
- **recall** = TP / (TP + FN): de los desbalances, ¿cuántos atrapé?
- **alertas** = el porcentaje de horas en que mandamos camión. Para el presupuesto, con `"todas"`, debe ser 20 % o menos.

In [ ]:
tn, fp, fn, tp = confusion_matrix(sel.y, alerta, labels=[0, 1]).ravel()   # primero lo que pasó, luego lo que dijo
tabla = pd.DataFrame([[tn, fp], [fn, tp]], index=["pasó calma", "pasó desbalance"],
                     columns=["dijo calma", "dijo alerta"])
ax = sns.heatmap(tabla, annot=True, fmt=",", cmap="Blues", cbar=False)
ax.set_title(f"Julio, estación {estacion}, umbral {umbral}")
ax.grid(False)
plt.show()

print(f"alertas:   {alerta.mean():.1%} de las horas ({tp + fp:,} camiones)")
print(f"precision: {precision_score(sel.y, alerta, zero_division=0):.2f}   (de cada 100 camiones, cuántos sirvieron)")
print(f"recall:    {recall_score(sel.y, alerta, zero_division=0):.2f}   (de cada 100 desbalances, a cuántos llegamos)")
print(f"desbalances atrapados: {tp:,}   perdidos: {fn:,}   viajes en balde: {fp:,}")

**¿Cuál es el umbral más bajo que cabe en el 20 %?** La tabla prueba varios umbrales con **todas** las estaciones.
Bajar el umbral manda más camiones: sube el recall y baja la precision.

In [ ]:
filas = []
for t in [0.20, 0.24, 0.25, 0.26, 0.27, 0.28, 0.30, 0.40, 0.50]:
    al = (julio.p >= t).astype(int)
    filas.append({"umbral": t, "alertas": round(al.mean(), 3),
                  "precision": round(precision_score(julio.y, al), 2),
                  "recall": round(recall_score(julio.y, al), 2),
                  "atrapados (TP)": int(((al == 1) & (julio.y == 1)).sum()),
                  "en balde (FP)": int(((al == 1) & (julio.y == 0)).sum()),
                  "cabe en 20 %": "sí" if al.mean() <= 0.20 else "no"})
pd.DataFrame(filas)

Escojan el umbral más bajo que cabe en el 20 % (la última columna dice "sí"), pónganlo en el paso 2
y prueben con su estación. Vean que el presupuesto es de toda la zona: en una estación muy movida,
el camión va casi todas las horas; en una tranquila, casi nunca.

In [ ]:
# Con el umbral que escogieron, ¿cuánto se lleva cada estación? (las 8 que más alertas reciben)
julio["alerta"] = (julio.p >= umbral).astype(int)
por_est = julio.groupby("station_id").agg(alertas=("alerta", "mean"), desbalances=("y", "mean")).round(2)
por_est.sort_values("alertas", ascending=False).head(8)

**1. En la tabla de umbrales, ¿qué quiere decir un recall de 0.53?**

- a) De cada 100 camiones, 53 sirvieron
- b) De cada 100 desbalances de julio, el camión llegó a 53
- c) El modelo acierta el 53 % de las horas
- d) Hay 53 % de horas con alerta

**Su respuesta (doble clic para escribir):** ___

**2. Operaciones consigue un segundo camión y ahora alcanza para más horas. ¿Qué hacen con el umbral?**

- a) Lo suben: más precision
- b) Lo bajan: más alertas, más recall y menos precision
- c) Lo dejan en 0.5
- d) Da igual, el modelo no cambia

**Su respuesta (doble clic para escribir):** ___

### Caza el error

Un compañero manda este reporte: *"Mi modelo tiene exactitud de 85 % en julio. Es buenísimo, ya lo podemos usar."*
Corran la celda: ¿qué modelo mandó?

In [ ]:
nunca = pd.Series(0, index=julio.index)   # la regla "nunca hay desbalance": no manda ni un camión
for nombre, al in [("nunca", nunca), ("logística, 0.5", (julio.p >= 0.5).astype(int)),
                   ("logística, 0.27", (julio.p >= 0.27).astype(int))]:
    print(f"{nombre:16s} exactitud {accuracy_score(julio.y, al):.1%}   recall {recall_score(julio.y, al):.2f}"
          f"   atrapados {int(((al == 1) & (julio.y == 1)).sum()):,}")

**3. ¿Qué le contestan al compañero de la exactitud de 85 %?**

- a) Que está muy bien
- b) Que 'nunca alertar' ya da 85.5 %: que reporte precision y recall
- c) Que suba el umbral
- d) Que use más estaciones

**Su respuesta (doble clic para escribir):** ___

### ¿Y si agregamos "el día del mes es par"?

Alguien propone una columna más: 1 si el día del mes es par, 0 si es non. No tiene por qué importar,
pero hay que comprobarlo con datos. Corran y miren dos cosas: el **peso** de la columna nueva y la **log-loss**
de julio (menos es mejor).

In [ ]:
for d in (train, julio):
    d["dia_par"] = (d.hour.dt.day % 2 == 0).astype(int)   # 1 = día par

con_par = LogisticRegression(max_iter=1000).fit(columnas(train).join(train.dia_par), train.y)
p_par = con_par.predict_proba(columnas(julio).join(julio.dia_par))[:, 1]

print(f"peso de dia_par: {con_par.coef_[0][-1]:.3f}   (el de x es {con_par.coef_[0][0]:.3f})")
print(f"log-loss julio sin dia_par: {log_loss(julio.y, julio.p):.4f}   con dia_par: {log_loss(julio.y, p_par):.4f}")

**4. La columna dia_par tiene peso 0.02 y la log-loss de julio no mejora. ¿Qué hacen?**

- a) La dejan: más columnas siempre es mejor
- b) La quitan: no ayuda a predecir
- c) Suben el umbral
- d) La pasan a one-hot

**Su respuesta (doble clic para escribir):** ___

## Paso 4. La respuesta a Operaciones

Copien la plantilla en la celda de abajo y llénenla con sus números. Tres renglones.

> **El número:** con umbral ___, mandamos camión en ___ % de las horas; llegamos a ___ de cada 100 desbalances
> (recall ___) y ___ de cada 100 viajes sirven (precision ___).
> **Qué tan seguro:** medido con las ___ horas de julio; en agosto y septiembre la proporción de desbalances puede cambiar.
> **Recomendación y riesgo:** recomendamos ___; el riesgo es ___.

*(Escriban aquí su respuesta.)*